# High-Level Backtesting of the Turtle Soup Strategy

In [1]:
# imports
import time
import pandas as pd
from nautilus_trader.backtest.config import BacktestVenueConfig, BacktestDataConfig, BacktestRunConfig
from nautilus_trader.backtest.engine import BacktestResult, BacktestEngine, BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.common.config import LoggingConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.model import BarType, Bar, Venue, InstrumentId
from nautilus_trader.model.enums import OmsType
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.config import DataCatalogConfig
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.trading.config import ImportableStrategyConfig
import sys
from pathlib import Path

from core.enums import MoneyManagementType

In [2]:
sys.path.append(str(Path.cwd().parent))

catalog = ParquetDataCatalog("../catalog")

start_ns = dt_to_unix_nanos(pd.Timestamp("2025-06-01"))
end_ns = dt_to_unix_nanos(pd.Timestamp("2025-10-22"))

instrument = TestInstrumentProvider.es_future(2025, 12)
instrument_id = instrument.id.value

# Configure backtesting
venue = BacktestVenueConfig(
    name="GLBX",
    oms_type=OmsType.NETTING,
    account_type="MARGIN",
    base_currency="USD",
    starting_balances=["10_000 USD"],
)

# Configure a catalog for a live system
catalog_cfg = DataCatalogConfig(
    path=str(catalog.path),
    fs_protocol="file",
    name="local"
)

base_bar_type = BarType.from_str(f"{instrument_id}-1-MINUTE-LAST-EXTERNAL")
data = BacktestDataConfig(
    catalog_path=str(catalog.path),
    catalog_fs_protocol="file",
    data_cls=Bar,
    bar_types=[base_bar_type],
    instrument_id=instrument_id,
    start_time=start_ns,
    end_time=end_ns
)

engine = BacktestEngineConfig(
    strategies=[
        ImportableStrategyConfig(
            strategy_path="strategies.turtle_soup.turtle_soup_strategy:TurtleSoupStrategy",
            config_path="strategies.turtle_soup.turtle_soup_strategy:TurtleSoupStrategyConfig",
            config={
                "liquidity_pool_bar_type": BarType.from_str(f"{instrument_id}-1-DAY-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "liquidity_pool_upper_period_window": 3,
                "liquidity_pool_lower_period_window": 3,

                "turtle_soup_bar_type": BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "turtle_soup_analysis_chain_bar_type": [BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                                                        BarType.from_str(f"{instrument_id}-15-MINUTE-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                                                        BarType.from_str(f"{instrument_id}-5-MINUTE-LAST-INTERNAL@1-MINUTE-EXTERNAL")],
                "retries_count_on_stop_out": 2,
                "turtle_soup_bars_count": 4,

                "risk_reward_ratio": 2.0,

                "money_management_type": MoneyManagementType.MIN_QUANTITY,
                "fixed_lot": 1,
                "fixed_risk_percent": 1,

                "instrument_id": instrument_id,
                "base_bar_type": base_bar_type,
                "is_backtest": True,
            },
        ),
    ],
    logging=LoggingConfig(log_level="ERROR"),
    catalogs=[catalog_cfg]
)

config = BacktestRunConfig(
    engine=engine,
    venues=[venue],
    data=[data],
)

node = BacktestNode(configs=[config])

# run backtesting
elapsed_start = time.perf_counter()
# Runs one or many configs synchronously
results: list[BacktestResult] = node.run()
elapsed_end = time.perf_counter()

print(f"Elapsed time: {elapsed_end - elapsed_start:.6f} seconds")

Elapsed time: 11.067704 seconds


In [3]:
results

[BacktestResult(trader_id='BACKTESTER-001', machine_id='Yuriis-MacBook-Pro.local', run_config_id='1bda480b5cb16c582c542028b2f2d3f4cc18b239ea690bcd82e72bd27c3cf87b', instance_id='6fb8240c-3b0a-44af-a617-dd41a99d2223', run_id='854f60aa-1a52-4ca3-a0c2-475983d04ed4', run_started=1762081437243627000, run_finished=1762081447781308000, backtest_start=1748815260000000000, backtest_end=1760918400000000000, elapsed_time=12103140.0, iterations=137138, total_events=1237, total_orders=465, total_positions=1, stats_pnls={'USD': {'PnL (total)': 96.26, 'PnL% (total)': 0.9626000000000021, 'Max Winner': 190.76, 'Avg Winner': 26.52914285714286, 'Min Winner': 2.0, 'Min Loser': 0.0, 'Avg Loser': -8.949032258064516, 'Max Loser': -59.51, 'Expectancy': 0.7520312500000004, 'Win Rate': 0.2734375}}, stats_returns={'Returns Volatility (252 days)': 0.046307543681948025, 'Average (Return)': -5.9998846713832827e-05, 'Average Loss (Return)': -0.001131932081592341, 'Average Win (Return)': 0.00275765422782396, 'Sharpe 

In [4]:
backtest_engine: BacktestEngine = node.get_engine(config.id)
positions = backtest_engine.trader.generate_positions_report()

In [5]:
len(positions)

128

In [6]:
pd.set_option("display.max_rows", 200)   # show all rows
pd.set_option("display.max_columns", None)  # show all cols
positions

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,ts_init,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
ESZ5.GLBX-TurtleSoupStrategy-000-14ba3afe-66c9-4ae3-afbf-a6d1042b1d77,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250602-000000-001-000-1,O-20250602-000000-001-000-2,SELL,FLAT,0,1,1748822400000000000,2025-06-02 00:00:00+00:00,1748888400000000000,2025-06-02 18:20:00+00:00,6.600000e+13,5898.750000,5932.750000,[0.00 USD],-0.00576,-34.00 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-85d9f40a-10be-4a88-8d57-6a969693b800,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250603-143500-001-000-4,O-20250603-143500-001-000-5,SELL,FLAT,0,1,1748961300000000000,2025-06-03 14:35:00+00:00,1748962080000000000,2025-06-03 14:48:00+00:00,7.800000e+11,5954.000000,5959.250000,[0.00 USD],-0.00088,-5.25 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-0c7fddbb-b1ab-4cdc-9f79-223043bcd0d3,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250604-090000-001-000-7,O-20250604-090000-001-000-8,SELL,FLAT,0,1,1749027660000000000,2025-06-04 09:01:00+00:00,1749031740000000000,2025-06-04 10:09:00+00:00,4.080000e+12,5990.000000,5996.000000,[0.00 USD],-0.00100,-6.00 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-7030f042-2241-442c-95c2-1fa73131a70e,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250605-125500-001-000-10,O-20250605-125500-001-000-12,SELL,FLAT,0,1,1749128100000000000,2025-06-05 12:55:00+00:00,1749147480000000000,2025-06-05 18:18:00+00:00,1.938000e+13,5994.500000,5950.500000,[0.00 USD],0.00734,44.00 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-266c29be-cb01-4fdd-8c7b-02078352f3a7,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250606-141500-001-000-13,O-20250606-141500-001-000-15,SELL,FLAT,0,1,1749219300000000000,2025-06-06 14:15:00+00:00,1749223140000000000,2025-06-06 15:19:00+00:00,3.840000e+12,6016.000000,5998.000000,[0.00 USD],0.00299,18.00 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-0e9bd45b-adba-4b24-804f-d36de54193cd,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250609-024500-001-000-16,O-20250609-024500-001-000-17,BUY,FLAT,0,1,1749437100000000000,2025-06-09 02:45:00+00:00,1749438840000000000,2025-06-09 03:14:00+00:00,1.740000e+12,5995.500000,5992.750000,[0.00 USD],-0.00046,-2.75 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-3803780b-dd28-41fa-8f58-8de9724de58f,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250609-083000-001-000-19,O-20250609-083000-001-000-20,SELL,FLAT,0,1,1749457800000000000,2025-06-09 08:30:00+00:00,1749459240000000000,2025-06-09 08:54:00+00:00,1.440000e+12,6008.500000,6013.750000,[0.00 USD],-0.00087,-5.25 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-3acb0235-1cab-4d4d-a4d9-1e575925aeaf,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250609-090000-001-000-22,O-20250609-090000-001-000-23,BUY,FLAT,0,1,1749459660000000000,2025-06-09 09:01:00+00:00,1749475920000000000,2025-06-09 13:32:00+00:00,1.626000e+13,6014.500000,6005.750000,[0.00 USD],-0.00145,-8.75 USD,True
ESZ5.GLBX-TurtleSoupStrategy-000-364b11b6-3958-4c49-8043-573273510e20,BACKTESTER-001,TurtleSoupStrategy-000,ESZ5.GLBX,GLBX-001,O-20250609-175000-001-000-25,O-20250609-175000-001-000-26,SELL,FLAT,0,1,1749491400000000000,2025-06-09 17:50:00+00:00,1749491460000000000,2025-06-09 17:51:00+00:00,6.000000e+10,6024.500000,6026.000000,[0.00 USD],-0.00025,-1.50 USD,True
